## **PHL-Correct-Proteins-OpenSWATH**

In [1]:
import pandas as pd
import polars as pl

#### **Load DIA-NN and OpenSWATH Libraries**

In [2]:
lib_diann_reannotated = pd.read_parquet("../../results/PanHuman-Library-Creation/test.parquet")

In [3]:
oswLib = pd.read_csv("../../results/K562-Refine-PanHuman-Lib-With-GPF/osw_tl/2025-09-19-PHL-lib-TL-On-GPF-OSW_fix_mods.tsv", sep='\t')

In [4]:
oswLib

,ModifiedPeptideSequence,PrecursorCharge,NormalizedRetentionTime,PrecursorIonMobility,PeptideSequence,PrecursorMz,ProteinId,GeneName,FragmentType,ProductMz,RelativeIntensity,ProductCharge,FragmentSeriesNumber
0,LIFDNLK,2,53.588224,0.779351,LIFDNLK,431.7553,6/P20648/Q13733/P13637/P50993/P54707/P05023,NaN,y,636.33514,1.000000,1,5
1,LIFDNLK,2,53.588224,0.779351,LIFDNLK,431.7553,6/P20648/Q13733/P13637/P50993/P54707/P05023,NaN,y,749.41920,0.253908,1,6
2,LIFDNLK,2,53.588224,0.779351,LIFDNLK,431.7553,6/P20648/Q13733/P13637/P50993/P54707/P05023,NaN,y,489.26672,0.138995,1,4
3,LIFDNLK,2,53.588224,0.779351,LIFDNLK,431.7553,6/P20648/Q13733/P13637/P50993/P54707/P05023,NaN,y,374.23980,0.092396,1,3
4,LIFDNLK,2,53.588224,0.779351,LIFDNLK,431.7553,6/P20648/Q13733/P13637/P50993/P54707/P05023,NaN,b,603.31366,0.016310,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2693924,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,5,-13.820642,1.097049,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,995.3931,1/P35527,NaN,b,968.37040,0.276863,1,13
2693925,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,5,-13.820642,1.097049,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,995.3931,1/P35527,NaN,b,1620.12230,0.152982,2,44
2693926,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,5,-13.820642,1.097049,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,995.3931,1/P35527,NaN,b,623.24200,0.124282,1,8
2693927,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,5,-13.820642,1.097049,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,995.3931,1/P35527,NaN,y,786.32510,0.063708,2,21


In [5]:
oswLib['Precursor'] = oswLib['ModifiedPeptideSequence'] + oswLib['PrecursorCharge'].astype(str)

In [6]:
len(oswLib['Precursor'].drop_duplicates())

204563

In [7]:
len(lib_diann_reannotated['Precursor.Id'].drop_duplicates())

204564

In [8]:
lib_diann_reannotated_precs = lib_diann_reannotated[['Precursor.Id', 'Protein.Ids']].drop_duplicates().rename(columns={'Precursor.Id':'Precursor'})

See that practically the same number of peptide precursors (DIA-NN reannotated has one more) suggesting that the DIA-NN library should be able to annotate this properly. 

In [9]:
oswLib.columns

Index(['ModifiedPeptideSequence', 'PrecursorCharge', 'NormalizedRetentionTime',
       'PrecursorIonMobility', 'PeptideSequence', 'PrecursorMz', 'ProteinId',
       'GeneName', 'FragmentType', 'ProductMz', 'RelativeIntensity',
       'ProductCharge', 'FragmentSeriesNumber', 'Precursor'],
      dtype='object')

In [10]:
out = oswLib.drop(columns='ProteinId').merge(lib_diann_reannotated_precs[['Precursor', 'Protein.Ids']], how='inner', on='Precursor').drop_duplicates().rename(columns={'Protein.Ids':'ProteinId'})

In [11]:
out

,ModifiedPeptideSequence,PrecursorCharge,NormalizedRetentionTime,PrecursorIonMobility,PeptideSequence,PrecursorMz,GeneName,FragmentType,ProductMz,RelativeIntensity,ProductCharge,FragmentSeriesNumber,Precursor,ProteinId
0,LIFDNLK,2,53.588224,0.779351,LIFDNLK,431.7553,NaN,y,636.33514,1.000000,1,5,LIFDNLK2,P05023;P13637;P20648;P50993;P54707;Q13733
1,LIFDNLK,2,53.588224,0.779351,LIFDNLK,431.7553,NaN,y,749.41920,0.253908,1,6,LIFDNLK2,P05023;P13637;P20648;P50993;P54707;Q13733
2,LIFDNLK,2,53.588224,0.779351,LIFDNLK,431.7553,NaN,y,489.26672,0.138995,1,4,LIFDNLK2,P05023;P13637;P20648;P50993;P54707;Q13733
3,LIFDNLK,2,53.588224,0.779351,LIFDNLK,431.7553,NaN,y,374.23980,0.092396,1,3,LIFDNLK2,P05023;P13637;P20648;P50993;P54707;Q13733
4,LIFDNLK,2,53.588224,0.779351,LIFDNLK,431.7553,NaN,b,603.31366,0.016310,1,5,LIFDNLK2,P05023;P13637;P20648;P50993;P54707;Q13733
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2693924,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,5,-13.820642,1.097049,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,995.3931,NaN,b,968.37040,0.276863,1,13,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,P35527
2693925,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,5,-13.820642,1.097049,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,995.3931,NaN,b,1620.12230,0.152982,2,44,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,P35527
2693926,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,5,-13.820642,1.097049,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,995.3931,NaN,b,623.24200,0.124282,1,8,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,P35527
2693927,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,5,-13.820642,1.097049,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,995.3931,NaN,y,786.32510,0.063708,2,21,GGSGGSYGGGSGSGGGSGGGYGGGSGGGHSGGSGGGHSGGSGGNYG...,P35527


In [12]:
out.to_csv("formattedLib_osw_tl/2025-09-19-PHL-lib-TL-On-GPF-OSW_fix_mods-new-prot-annot.tsv", sep='\t', index=False)